In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,GridSearchCV,cross_val_predict
from xgboost import XGBClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)


In [2]:
df=pd.read_csv("../Data/ai4i2020.csv")


In [3]:
df.drop(columns=["UDI","Product ID"],inplace=True)

df["Power"]=(df["Torque [Nm]"]*df["Rotational speed [rpm]"])

type_mapping={
    "L":0,
    "M":1,
    "H":2
}

df["Type"]=df["Type"].map(type_mapping)

In [4]:
target="TWF"
failure_columns=[
    "Machine failure",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
    "TWF"
]


In [5]:
X=df.drop(columns=failure_columns)
y=df[target]

X.columns = (
    X.columns
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("<", "", regex=False)
)

In [6]:
X_train,X_test,y_train,y_test=train_test_split(
    X,y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [7]:
xgb=XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=1
)

param_grid={
    "n_estimators":[100,200,300],
    "max_depth":[3,5,7],
    "learning_rate":[0.03,0.05,0.1],
    "subsample":[0.8,1.0],
    "colsample_bytree":[0.8,1.0]
}


grid=GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train,y_train)

print("Best parameters:")
print(grid.best_params_)

print("Best CV F1:")
print(grid.best_score_)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best CV F1:
0.0


In [9]:

best_model = grid.best_estimator_

test_probs = best_model.predict_proba(X_test)[:, 1]

test_preds = (test_probs >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_preds,
    labels=[0, 1]
).ravel()

accuracy = accuracy_score(y_test, test_preds)
precision = precision_score(y_test, test_preds, zero_division=0)
recall = recall_score(y_test, test_preds, zero_division=0)
f1 = f1_score(y_test, test_preds, zero_division=0)
roc_auc = roc_auc_score(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)

print("=" * 60)
print("XGBOOST — TWF")
print("=" * 60)

print("Best parameters:")
print(grid.best_params_)

print(f"\nCV F1: {grid.best_score_:.4f}")

print("\nTest confusion matrix:")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TP: {tp}")

print("\nTest metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("=" * 60)

XGBOOST — TWF
Best parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}

CV F1: 0.0000

Test confusion matrix:
TN: 2986
FP: 0
FN: 14
TP: 0

Test metrics:
Accuracy : 0.9953
Precision: 0.0000
Recall   : 0.0000
F1       : 0.0000
ROC-AUC  : 0.9767
PR-AUC   : 0.1743


In [10]:
cv_probs = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

thresholds=np.arange(0.01,1.0,0.01)

best_threshold=0
best_f1=0

for threshold in thresholds:
    preds=(cv_probs>=threshold).astype(int)

    precision = precision_score(
        y_train,
        preds,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        preds,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        preds,
        zero_division=0
    )

    tn,fp,fn,tp=confusion_matrix(
        y_train,
        preds,
        labels=[0,1]
    ).ravel()

    print(
        f"{threshold:.2f}"
        f"{precision:.4f}"
        f"{recall:.4f}"
        f"{f1:.4f}"
        f"{fp}"
        f"{fn}"
    )

    if f1>best_f1:
        best_f1=f1
        best_threshold=threshold

print("\nSelected threshold:",best_threshold)
print("Best OOF F1:",best_f1)



0.010.05480.87500.10314834
0.020.05490.68750.101637910
0.030.05730.56250.104029614
0.040.06580.46880.115421317
0.050.05390.28120.090515823
0.060.04920.18750.077911626
0.070.03370.09380.04968629
0.080.02820.06250.03886930
0.090.01820.03120.02305431
0.100.02130.03120.02534631
0.110.03030.03120.03083231
0.120.03850.03120.03452531
0.130.00000.00000.00002032
0.140.00000.00000.00001732
0.150.00000.00000.00001332
0.160.00000.00000.0000932
0.170.00000.00000.0000732
0.180.00000.00000.0000732
0.190.00000.00000.0000532
0.200.00000.00000.0000232
0.210.00000.00000.0000132
0.220.00000.00000.0000132
0.230.00000.00000.0000132
0.240.00000.00000.0000132
0.250.00000.00000.0000132
0.260.00000.00000.0000132
0.270.00000.00000.0000132
0.280.00000.00000.0000032
0.290.00000.00000.0000032
0.300.00000.00000.0000032
0.310.00000.00000.0000032
0.320.00000.00000.0000032
0.330.00000.00000.0000032
0.340.00000.00000.0000032
0.350.00000.00000.0000032
0.360.00000.00000.0000032
0.370.00000.00000.0000032
0.380.00000.00000.

In [11]:
best_model = grid.best_estimator_

test_probs = best_model.predict_proba(X_test)[:, 1]

threshold = 0.04

test_preds = (test_probs >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_preds,
    labels=[0, 1]
).ravel()

accuracy = accuracy_score(y_test, test_preds)
precision = precision_score(y_test, test_preds, zero_division=0)
recall = recall_score(y_test, test_preds, zero_division=0)
f1 = f1_score(y_test, test_preds, zero_division=0)
roc_auc = roc_auc_score(y_test, test_probs)
pr_auc = average_precision_score(y_test, test_probs)

print("=" * 60)
print("XGBOOST — TWF — OOF SELECTED THRESHOLD")
print("=" * 60)

print(f"Threshold: {threshold:.2f}")

print("\nConfusion matrix:")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TP: {tp}")

print("\nTest metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("=" * 60)

XGBOOST — TWF — OOF SELECTED THRESHOLD
Threshold: 0.04

Confusion matrix:
TN: 2882
FP: 104
FN: 4
TP: 10

Test metrics:
Accuracy : 0.9640
Precision: 0.0877
Recall   : 0.7143
F1       : 0.1562
ROC-AUC  : 0.9767
PR-AUC   : 0.1743
